# as-strided-windowing — faded example 1: 1-D max-pool via strided windows (complete the stride tuple)

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `as-strided-windowing`. Running the beacon reports progress on the `PyTorch: as_strided windowing` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: as_strided windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`as-strided-windowing`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "as-strided-windowing"
DD_SUBTOPIC = "PyTorch: as_strided windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A strided 1-D pooling window of width `K` and step `step` produces rows `x[i*step : i*step + K]`. With `as_strided`, the **outer** stride is `s*step` (jump `step` elements between windows) and the **inner** stride is `s` (walk within a window). Reducing the window view with `.amax(dim=-1)` gives 1-D max pooling.

## Faded exercise 1

### Faded — 1-D max pooling with stride via as_strided

Implement `max_pool_1d(x, K, step)` for a 1-D tensor `x`. Build a zero-copy `(L_out, K)` window view where row `i` is `x[i*step : i*step + K]` with `L_out = (L - K) // step + 1`, then take the max over each window so the result has shape `(L_out,)`.

The `size`, the output length, and the final reduction are already written. **You must complete the `stride` argument** passed to `t.as_strided`.

**Fill in:** The (outer, inner) stride tuple for the window view: outer steps `step` source elements between windows, inner steps one source element within a window.

In [ ]:
def max_pool_1d(x: Tensor, K: int, step: int) -> Tensor:
    L = x.shape[0]
    s, = x.stride()
    L_out = (L - K) // step + 1
    stride = (s * step, s)
    windows = t.as_strided(x, size=(L_out, K), stride=stride)
    return windows.amax(dim=-1)


def _test():
    t.manual_seed(0)
    x = t.randn(13)
    for K, step in [(3, 1), (3, 2), (4, 3), (2, 5)]:
        out = max_pool_1d(x, K, step)
        L_out = (x.shape[0] - K) // step + 1
        assert out.shape == (L_out,), (out.shape, L_out)
        ref = t.stack([x[i * step:i * step + K].max() for i in range(L_out)])
        assert t.allclose(out, ref), (out, ref)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def max_pool_1d(x: Tensor, K: int, step: int) -> Tensor:
    L = x.shape[0]
    s, = x.stride()
    L_out = (L - K) // step + 1
    stride = (s * step, s)
    windows = t.as_strided(x, size=(L_out, K), stride=stride)
    return windows.amax(dim=-1)
```
</details>